# Practical ML II: Debugging, Feature Engineering & Big-Data Spark

**Follow-up to Practical ML I** · Intermediate → advanced · Laptop-friendly

**By the end:** diagnose leakage and overfitting; engineer features without leaking; tune and calibrate classifiers; compare learning curves; understand Spark partitions, shuffles, skew and joins; benchmark Spark vs pandas fairly; build a distributed ML pipeline; and monitor drift.

**How to use:** Run cells in order. Read each **Interpretation** prompt *before* looking at its chart. Spark labs are optional and gracefully skipped when Spark/Java is unavailable. Synthetic datasets make results reproducible and avoid external downloads.

> **Important:** Spark on one laptop is still Spark, but it is *not* a multi-node speed test. Tiny benchmarks are educational, not universal performance claims.

## Setup
From your project directory:

```bash
uv add numpy pandas matplotlib scikit-learn scipy jupyterlab pyarrow pyspark
uv run jupyter lab
```

PySpark requires a compatible Java runtime. Check your installed PySpark version's Java requirements; many recent releases support Java 17+. If Spark cannot start, all sklearn labs still work.

**Run time:** Core labs ~15–40 minutes; Spark benchmarks vary. Reduce `N_SPARK` if memory is limited.

In [ ]:
import sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, learning_curve, RandomizedSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, SGDClassifier, SGDRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report, ConfusionMatrixDisplay,
                             roc_auc_score, average_precision_score, precision_recall_curve, brier_score_loss,
                             log_loss, mean_absolute_error)
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV
from sklearn.inspection import permutation_importance
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Python:',sys.version.split()[0], '| sklearn, numpy and pandas ready')

# Part I — Know when a model is doing nothing
A **baseline** is a deliberately simple predictor. A model should improve on it *out of sample*, using the metric that reflects your actual task.

**Trap:** With 95% negatives, always predicting negative is 95% accurate but catches none of the positives.

In [ ]:
rng = np.random.default_rng(42)
X, y = make_classification(n_samples=4500, n_features=15, n_informative=5, n_redundant=3,
                           weights=[.95, .05], flip_y=.01, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
models = {'Do-nothing baseline': DummyClassifier(strategy='most_frequent'),
          'Logistic regression': Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=1200,class_weight='balanced'))]),
          'Random forest': RandomForestClassifier(n_estimators=120, min_samples_leaf=3,random_state=42,n_jobs=-1)}
rows=[]; fitted={}
for name, model in models.items():
    model.fit(X_train,y_train); fitted[name]=model
    score = model.predict_proba(X_test)[:,1]
    pred = model.predict(X_test)
    rows.append({'Model':name,'Accuracy':accuracy_score(y_test,pred), 'Balanced accuracy':balanced_accuracy_score(y_test,pred),
                 'PR-AUC':average_precision_score(y_test,score),'ROC-AUC':roc_auc_score(y_test,score)})
results = pd.DataFrame(rows).set_index('Model')
display(results.round(3))
print('Positive prevalence:',round(y_test.mean(),3), '| Random PR-AUC baseline is approximately prevalence')

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(12,3.3))
for ax,(name,model) in zip(axes,fitted.items()):
    ConfusionMatrixDisplay.from_estimator(model,X_test,y_test,ax=ax,colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

**Interpretation:** Accuracy can conceal total failure on a minority class. **PR-AUC** summarizes precision–recall across thresholds; its random-ranking baseline is approximately positive-class prevalence. ROC-AUC answers a different ranking question. Inspect the confusion matrix *and* whether false positives or false negatives cost more.

**Try:** Remove `class_weight='balanced'`. Does accuracy improve while recall drops? Why?

# Part II — Manufacture and catch data leakage
**Leakage** occurs when a feature contains information unavailable at prediction time, or when training procedures learn from validation/test data. It can produce impressive metrics that collapse in production.

In [ ]:
rng = np.random.default_rng(7)
X0, y0 = make_classification(n_samples=2500,n_features=8,n_informative=4,random_state=7)
X_leaky = np.column_stack([X0, y0 + rng.normal(0,.03,len(y0))])  # fake 'outcome recorded later'
a,b,ya,yb = train_test_split(X_leaky,y0,test_size=.3,stratify=y0,random_state=7)
for label, cols in [('Legitimate features',slice(None,-1)),('With leaked future outcome',slice(None,None))]:
    model = RandomForestClassifier(n_estimators=80,random_state=7,n_jobs=-1).fit(a[:,cols],ya)
    print(label, 'ROC-AUC:', round(roc_auc_score(yb,model.predict_proba(b[:,cols])[:,1]),3))

**Interpretation:** The leak is intentionally extreme. Ask of **every feature**: *Was it known at the exact time the prediction would be made?* Other leakage: standardizing the full dataset before splitting; target encoding before CV; repeated users appearing in both train/test; and random splits for future forecasting.

**Exercise:** Describe a leakage feature in fraud detection, and how to remove it.

# Part III — Train/test gaps, tree depth, and learning curves
**Underfitting:** both training and validation scores are poor. **Overfitting:** training is strong, validation is weak. A **learning curve** varies the amount of training data to show whether more examples might help.

In [ ]:
Xc,yc = make_classification(n_samples=1700,n_features=20,n_informative=5,n_redundant=4,flip_y=.12,random_state=25)
xa,xb,ya,yb=train_test_split(Xc,yc,test_size=.3,stratify=yc,random_state=25)
depths=[1,2,3,4,5,7,10,15,None]; train_sc=[]; test_sc=[]
for depth in depths:
    tree=DecisionTreeClassifier(max_depth=depth,min_samples_leaf=2,random_state=42).fit(xa,ya)
    train_sc.append(roc_auc_score(ya,tree.predict_proba(xa)[:,1]))
    test_sc.append(roc_auc_score(yb,tree.predict_proba(xb)[:,1]))
plt.figure(figsize=(8,4)); xx=np.arange(len(depths))
plt.plot(xx,train_sc,'o-',label='Train ROC-AUC');plt.plot(xx,test_sc,'o-',label='Held-out ROC-AUC')
plt.xticks(xx,[str(x) for x in depths]);plt.xlabel('Tree max_depth');plt.ylabel('ROC-AUC');plt.legend();plt.grid(alpha=.2);plt.show()

In [ ]:
sizes,tr,cv=learning_curve(DecisionTreeClassifier(max_depth=5,min_samples_leaf=4,random_state=42),
    Xc,yc,train_sizes=np.linspace(.1,1,7),cv=StratifiedKFold(4,shuffle=True,random_state=42),
    scoring='roc_auc',n_jobs=-1)
plt.figure(figsize=(7,4));plt.plot(sizes,tr.mean(axis=1),'o-',label='Train');plt.plot(sizes,cv.mean(axis=1),'o-',label='Cross-validation')
plt.xlabel('Training examples');plt.ylabel('ROC-AUC');plt.legend();plt.grid(alpha=.2);plt.show()

**Interpretation:** A widening gap suggests variance/overfitting; two low, close curves suggest bias/underfitting. A validation curve still rising with sample size suggests more data *may* help. Noise, leakage and distribution shifts can mislead these diagnoses.

**Challenge:** Re-run with `max_depth=1`, then `max_depth=None`. Compare patterns.

# Part IV — Feature engineering without test contamination
Use a **Pipeline** so imputation, scaling and feature transformations are fitted only on training folds. Scaling matters for distance/regularized-linear models, generally not for trees.

In [ ]:
rng=np.random.default_rng(40); m=2200
age=rng.integers(18,75,m); income=np.exp(rng.normal(10.5,.6,m)); visits=rng.poisson(3,m)
channel=rng.choice(['organic','search','referral'],size=m,p=[.45,.35,.2])
logits=-6 + .055*age + .000025*income + .33*visits + (channel=='referral')*.8
p=1/(1+np.exp(-logits)); target=rng.binomial(1,p)
df=pd.DataFrame({'age':age.astype(float),'income':income,'visits':visits.astype(float),'channel':channel})
df.loc[rng.choice(m,90,replace=False),'income']=np.nan
from sklearn.preprocessing import OneHotEncoder
num=['age','income','visits'];cat=['channel']
pre=ColumnTransformer([('numerical',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
                       ('categorical',OneHotEncoder(handle_unknown='ignore'),cat)])
pipe=Pipeline([('prep',pre),('classifier',LogisticRegression(max_iter=1000))])
xtr,xte,ytr,yte=train_test_split(df,target,test_size=.25,stratify=target,random_state=42)
pipe.fit(xtr,ytr)
print('Test ROC-AUC:',round(roc_auc_score(yte,pipe.predict_proba(xte)[:,1]),3))

**Try:** Add `income_per_visit = income / (visits + 1)` to a copy of the dataframe *before* splitting (this is row-wise, independent of outcomes). Update `num` and re-fit the pipeline. For transformations depending on population statistics or target labels, fit them **inside** each training fold.

# Part V — Tune hyperparameters responsibly
**Hyperparameters** set model behavior before learning (e.g., tree depth). Randomized search explores a distribution of candidate settings. Keep a final **test set untouched** until model selection is complete.

In [ ]:
search_pipe=Pipeline([('impute',SimpleImputer()),('rf',RandomForestClassifier(random_state=42,n_jobs=1))])
param_grid={'rf__n_estimators':[60,100,150], 'rf__max_depth':[3,5,9,None],
            'rf__min_samples_leaf':[1,3,8,16], 'rf__max_features':['sqrt',.7]}
tuner=RandomizedSearchCV(search_pipe,param_grid,n_iter=12,scoring='average_precision',cv=StratifiedKFold(3,shuffle=True,random_state=42),
                         n_jobs=-1,random_state=42,return_train_score=True)
tuner.fit(X_train,y_train)
print('Best CV PR-AUC:',round(tuner.best_score_,3));print('Params:',tuner.best_params_)
print('FINAL test PR-AUC:',round(average_precision_score(y_test,tuner.predict_proba(X_test)[:,1]),3))

In [ ]:
cvdf=pd.DataFrame(tuner.cv_results_)
plt.figure(figsize=(7,4));plt.scatter(cvdf['mean_train_score'],cvdf['mean_test_score'],c=cvdf['param_rf__min_samples_leaf'].astype(float))
plt.xlabel('CV train PR-AUC');plt.ylabel('CV validation PR-AUC');plt.title('Each dot = tuning candidate; color = minimum leaf size')
plt.colorbar(label='min_samples_leaf');plt.grid(alpha=.2);plt.show()

**Interpretation:** The best cross-validation score is optimistically biased because you chose the maximum. Treat final test performance as the more credible estimate. For further iteration after reading test results, create a *new* untouched test set (or use nested CV).

**Exercise:** What happens when `min_samples_leaf` gets large?

# Part VI — Thresholds and calibrated probabilities
A ranking model outputs scores. A **decision threshold** converts scores into yes/no actions. The best threshold depends on your costs and capacity, not a universal 0.5. A **calibrated** model's predicted 0.7 cases should succeed around 70% of the time over many cases.

In [ ]:
scores=fitted['Random forest'].predict_proba(X_test)[:,1]
precision,recall,thresholds=precision_recall_curve(y_test,scores)
f1=2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1],1e-12)
best_idx=np.argmax(f1)
print('F1-optimal threshold ON TEST (demonstration only):',round(thresholds[best_idx],3))
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].plot(thresholds,precision[:-1],label='Precision');axs[0].plot(thresholds,recall[:-1],label='Recall');axs[0].set_xlabel('Threshold');axs[0].legend();axs[0].grid(alpha=.2)
CalibrationDisplay.from_predictions(y_test,scores,n_bins=8,ax=axs[1]);axs[1].set_title('Calibration of untuned random forest')
plt.tight_layout();plt.show()
print('Brier score (lower better):',round(brier_score_loss(y_test,scores),4))

**Critical warning:** We selected the best threshold on the **test** labels only to visualize the method. **Never report that resulting test F1 as unbiased.** In real projects, choose your threshold on a separate validation set or out-of-fold predictions, then evaluate it once on untouched test data.

**Exercise:** If false negatives cost $100 and false positives $5, write a cost function for a threshold search on validation predictions.

# Part VII — Feature importance can mislead
Impurity-based tree importance can favor high-cardinality variables. **Permutation importance** instead shuffles one column on held-out data and observes the drop in the scoring metric. Correlated predictors can hide each other's importance.

In [ ]:
rf=RandomForestClassifier(n_estimators=100,min_samples_leaf=4,random_state=42,n_jobs=-1).fit(xa,ya)
imp=permutation_importance(rf,xb,yb,scoring='roc_auc',n_repeats=4,random_state=42,n_jobs=-1)
order=np.argsort(imp.importances_mean)[-10:]
plt.figure(figsize=(8,4));plt.barh(np.array([f'feature {i}' for i in range(Xc.shape[1])])[order],imp.importances_mean[order])
plt.xlabel('Mean decrease in ROC-AUC when shuffled');plt.title('Held-out permutation importance');plt.show()

# Part VIII — Time series, drift and incremental learning
A random split can allow the future to influence the past. **TimeSeriesSplit** is chronological, though real forecasting may also require a **gap/embargo** to avoid overlapping outcomes. **Drift** means the input distribution or input→outcome relationship changes over time.

In [ ]:
rng=np.random.default_rng(13); t=np.arange(1500)
feature=rng.normal(size=len(t)); regime=(t>=1000)
y_ts=(feature + rng.normal(0,.8,len(t)) > np.where(regime,1.3,-.4)).astype(int)
# One-feature model, trained only on the early regime
cut=950
historical=LogisticRegression().fit(feature[:cut].reshape(-1,1),y_ts[:cut])
chunks=[]
for start in range(cut,len(t),110):
    end=min(start+110,len(t)); pr=historical.predict(feature[start:end].reshape(-1,1))
    chunks.append((start,accuracy_score(y_ts[start:end],pr),y_ts[start:end].mean()))
watch=pd.DataFrame(chunks,columns=['Start','Accuracy','Positive rate']);display(watch.round(3))
fig,ax=plt.subplots(figsize=(8,4));ax.plot(watch.Start,watch.Accuracy,'o-',label='Accuracy');ax.plot(watch.Start,watch['Positive rate'],'o-',label='Observed positive rate');ax.legend();ax.set_xlabel('Time index');ax.grid(alpha=.2);plt.show()

**Interpretation:** The label prevalence changes after index 1000; a previously good classifier may fail even though its code is unchanged. In production also monitor missingness, feature distributions, calibration, and subgroup outcomes. Monitoring cannot prove the cause of degradation by itself.

In [ ]:
# Incremental learning: works for certain algorithms, not ordinary random forests.
sgd=SGDClassifier(loss='log_loss',random_state=42)
chunk=300
for i in range(0,len(X_train),chunk):
    sgd.partial_fit(X_train[i:i+chunk],y_train[i:i+chunk],classes=np.array([0,1]))
print('Incremental SGD PR-AUC:',round(average_precision_score(y_test,sgd.predict_proba(X_test)[:,1]),3))
print('Note: scaled features and shuffled/representative batches are important for real SGD.')

# Part IX — Data formats and local performance
Spark is useful when work can be distributed and data does not fit comfortably in one machine's memory. **Parquet** is compressed, columnar storage that supports efficient column reads. **Lazy evaluation** means Spark builds a plan and runs only when you ask for results (such as `count()` or `write`).

In [ ]:
# A local, repeatable pandas read/write demonstration (optional pyarrow installed).
import tempfile, pathlib
small = pd.DataFrame({'id':np.arange(100_000), 'group':rng.integers(0,15,100_000), 'value':rng.normal(size=100_000)})
with tempfile.TemporaryDirectory() as tmp:
    csv=pathlib.Path(tmp)/'sample.csv';pq=pathlib.Path(tmp)/'sample.parquet'
    small.to_csv(csv,index=False)
    try:
        small.to_parquet(pq,index=False)
        print('CSV MB:',round(csv.stat().st_size/1e6,2),'Parquet MB:',round(pq.stat().st_size/1e6,2))
        print('Only one Parquet column:',pd.read_parquet(pq,columns=['value']).shape)
    except ImportError:
        print('Install pyarrow for this lab: uv add pyarrow')

**Interpretation:** File size and speed depend on data types, compression, schema and hardware. This toy test is not a guaranteed performance ratio.

# Part X — PySpark under the hood

- **Driver:** coordinates your application; `collect()` brings data here and can crash it.
- **Executor:** worker process running tasks (in local mode everything runs on your laptop).
- **Partition:** chunk of distributed data, the basic unit of parallel work.
- **DAG:** directed acyclic graph; Spark plans stages of transformations before execution.
- **Shuffle:** redistribution across partitions (e.g., for a groupBy or join), often expensive.
- **Data skew:** one key/partition has far more rows than others, creating stragglers.

**One rule:** Use native Spark SQL functions; avoid Python UDFs where built-in operations exist. Never call `toPandas()` on unknown-size data.

In [ ]:
HAVE_SPARK=False
try:
    from pyspark.sql import SparkSession, functions as F
    from pyspark.ml import Pipeline as SparkPipeline
    from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkStandardScaler
    from pyspark.ml.classification import LogisticRegression as SparkLogisticRegression, RandomForestClassifier as SparkRandomForestClassifier
    from pyspark.ml.evaluation import BinaryClassificationEvaluator
    from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
    spark=SparkSession.builder.master('local[2]').appName('ML_II_laptop_labs').config('spark.sql.shuffle.partitions','4').getOrCreate()
    spark.sparkContext.setLogLevel('ERROR')
    HAVE_SPARK=True
    print('Spark ready:',spark.version)
except Exception as e:
    print('Spark unavailable; skipping all Spark labs. Reason:',str(e)[:350])

## Lab A — Lazy execution, shuffle, and the execution plan
Read the physical plan. Look for **Exchange** (often a shuffle). Repartitioning and grouping create work; an action actually triggers it.

In [ ]:
N_SPARK=100_000  # increase to 1_000_000 after testing memory
if HAVE_SPARK:
    sdf=spark.range(N_SPARK).withColumn('key',(F.col('id')%20).cast('int')).withColumn('x',(F.col('id')*.01).cast('double'))
    print('Partitions:',sdf.rdd.getNumPartitions())
    grouped=sdf.groupBy('key').agg(F.avg('x').alias('mean_x'))
    grouped.explain(mode='formatted')
    print('Action result:',grouped.count())
else: print('SKIPPED: Spark unavailable')

**Interpretation:** `explain()` displays the physical plan; `Exchange` shows data redistribution. Spark's adaptive query execution may change the executed plan from the initial plan. The local Spark UI is usually at `http://localhost:4040` while the session runs; inspect jobs, stages, tasks and skew there.

## Lab B — Partitions, caching, and skew
A fair benchmark should **warm up**, run multiple times, and compare repeated medians; don't extrapolate laptop timings to a cluster. Cache only data that will be reused, then unpersist it.

In [ ]:
if HAVE_SPARK:
    def seconds(fn, repeats=2):
        measurements=[]
        for _ in range(repeats):
            start=time.perf_counter();fn();measurements.append(time.perf_counter()-start)
        return round(float(np.median(measurements)),3)
    a=sdf.repartition(2);b=sdf.repartition(8)
    print('2 partitions: ',seconds(lambda:a.groupBy('key').count().count()))
    print('8 partitions: ',seconds(lambda:b.groupBy('key').count().count()))
    reused=sdf.filter('x > 100')
    print('Uncached action:',seconds(lambda:reused.count()))
    reused.cache(); reused.count()  # materialize cache
    print('Cached action:',seconds(lambda:reused.count()))
    reused.unpersist()
else: print('SKIPPED')

In [ ]:
if HAVE_SPARK:
    skewed=spark.range(N_SPARK).withColumn('key',F.when(F.col('id')%10 != 0,F.lit(0)).otherwise(F.col('id')%100))
    distribution=skewed.groupBy('key').count().orderBy(F.desc('count'))
    distribution.show(8)
    print('Inspect whether a few keys dominate and explain why operations grouping by key can straggle.')
else:print('SKIPPED')

**Exercise:** Explain how salting a hot key can distribute work for associative aggregations. **Caution:** salting isn't universally correct for every join/aggregation and adds complexity.

## Lab C — Join strategies and broadcast
A broadcast join sends a small table to workers rather than shuffling both sides. Inspect the plan: the optimizer may automatically choose broadcast depending on statistics and settings.

In [ ]:
if HAVE_SPARK:
    lookup=spark.createDataFrame([(i,'segment'+str(i)) for i in range(20)],['key','segment'])
    plain=sdf.join(lookup,'key')
    broadcasted=sdf.join(F.broadcast(lookup),'key')
    print('=== Broadcast join physical plan ==='); broadcasted.explain()
    print('=== Ordinary join physical plan ==='); plain.explain()
    print('Joined rows:',broadcasted.count())
else:print('SKIPPED')

## Lab D — Spark ML pipeline (classification)
Spark ML uses a feature **vector** column. Build features with `VectorAssembler`, fit estimators, and transform held-out data. This trains within a single-machine Spark session, not a remote cluster.

In [ ]:
if HAVE_SPARK:
    # Predict a synthetic classification label: no external downloads
    spdata=spark.range(N_SPARK).select('id',
        (F.sin(F.col('id')*.02)).alias('f1'),
        (F.cos(F.col('id')*.04)).alias('f2'),
        ((F.col('id')%17)/17).cast('double').alias('f3'))
    spdata=spdata.withColumn('label',((F.col('f1')+F.col('f2')+F.col('f3'))>0.4).cast('double'))
    train_sp,test_sp=spdata.randomSplit([.75,.25],seed=42)
    pipe_sp=SparkPipeline(stages=[VectorAssembler(inputCols=['f1','f2','f3'],outputCol='features'),
        SparkLogisticRegression(labelCol='label',featuresCol='features',maxIter=30)])
    model_sp=pipe_sp.fit(train_sp)
    scored_sp=model_sp.transform(test_sp)
    print('Spark test area under ROC:',BinaryClassificationEvaluator(labelCol='label',metricName='areaUnderROC').evaluate(scored_sp))
    scored_sp.select('label','prediction','probability').show(5,truncate=False)
else: print('SKIPPED')

## Lab E — Spark hyperparameter tuning
**Spark `CrossValidator`** searches an estimator parameter grid. Limit folds, parallelism and grid size on a laptop; distributed training can become expensive. As before, reserve a final test set.

In [ ]:
if HAVE_SPARK:
    assembler=VectorAssembler(inputCols=['f1','f2','f3'],outputCol='features')
    lr=SparkLogisticRegression(labelCol='label',featuresCol='features',maxIter=20)
    pipeline=SparkPipeline(stages=[assembler,lr])
    grid=ParamGridBuilder().addGrid(lr.regParam,[0.0,.1]).addGrid(lr.elasticNetParam,[0.0,.5]).build()
    cv=CrossValidator(estimator=pipeline,estimatorParamMaps=grid,
        evaluator=BinaryClassificationEvaluator(labelCol='label',metricName='areaUnderROC'),numFolds=3,parallelism=2,seed=42)
    tuned=cv.fit(train_sp)
    print('CV metric values:',[round(x,4) for x in tuned.avgMetrics])
    print('Held-out test ROC-AUC:',round(BinaryClassificationEvaluator(labelCol='label',metricName='areaUnderROC').evaluate(tuned.transform(test_sp)),4))
else:print('SKIPPED')

## Lab F — pandas vs Spark benchmark
Compare comparable transformations *on the same inputs*, excluding disk I/O and initialization. This benchmark is intentionally simplified. Pandas uses local in-memory arrays; Spark has scheduling overhead. The result depends on hardware, partitions and data size.

In [ ]:
N_BENCH=100_000
local=pd.DataFrame({'id':np.arange(N_BENCH,dtype=np.int64)})
local['key']=local.id%20;local['x']=local.id*.01
start=time.perf_counter()
for _ in range(3): ans_local=local.groupby('key',sort=False)['x'].mean()
pandas_s=(time.perf_counter()-start)/3
print('Pandas mean seconds over 3 trials:',round(pandas_s,4))
if HAVE_SPARK:
    common=spark.createDataFrame(local)
    spark_task=lambda: common.groupBy('key').agg(F.avg('x')).collect()
    spark_task() # warm-up
    spark_s=seconds(spark_task,repeats=3)
    print('Spark median seconds over 3 warmed runs:',spark_s)
    print('Spark / pandas:',round(spark_s/max(pandas_s,1e-10),1),'× (this machine and workload only)')
else:print('Spark benchmark skipped')

# Part XI — A production-oriented troubleshooting checklist
| Symptom | First check | Next step |
|---|---|---|
| Accuracy high but useful predictions absent | Positive prevalence, confusion matrix, baseline | PR-AUC; choose cost-sensitive threshold |
| Train excellent, validation poor | Overfitting, leakage, duplicates | Simplify; group-aware or time-aware splits |
| Both poor | Wrong features, too-simple model, label noise | Feature engineering, error inspection, learning curves |
| CV great, deployment bad | Time leakage, drift, changed data pipeline | Temporal holdout, data validation and monitoring |
| Spark slow for small data | Scheduling overhead | Use pandas / sklearn locally |
| Spark freezes at one stage | Skew, shuffle, join explosion | Spark UI, explain plan, repartition or broadcast |
| Driver crashes | `collect()`, `toPandas()` or oversized metadata | Keep computations distributed, sample/aggregate |
| Training inconsistent | Seed, partitions, data ordering, nondeterminism | Version and track seeds/data/model configs |

**Operational checklist:** define a meaningful target; choose a metric and baseline; establish train/validation/test chronology; fit all preprocessing inside CV; log data versions and parameters; inspect subgroup errors; keep test untouched; monitor drift and latency.

# Final capstone — Choose one

**A. Fraud detection (recommended):** Generate 100k events with only 0.5% fraud; add changing fraud patterns over time. Compare a dummy model, logistic regression and random forest. Make train/validation/test chronological, optimize expected cost on validation, and report test results with class prevalence. Explain false alarms.

**B. Distributed event analysis:** Generate 1M–5M simulated transactions in Spark. Write/read Parquet; aggregate by customer; experiment with 2/4/8 partitions; inspect joins and `explain()`, introduce a skewed key, then benchmark cache vs no cache. Document where Spark does and does not help.

**C. Book recommendation prototype:** Create user/book interactions. Compare popularity baseline, item co-occurrence, and Spark ALS (alternating least squares collaborative filtering). Hold out interactions **per user**; evaluate ranking metrics such as Recall@K (top-K recommendation recall), and describe cold-start issues. Spark ALS is an optional additional module not implemented here.

## Self-check
1. Why is test-time threshold optimization leakage?
2. What would a learning curve look like for severe overfitting?
3. Why can a high-cardinality ID appear important in trees?
4. What does `Exchange` mean in Spark's plan?
5. When is `.toPandas()` dangerous?
6. Why might Spark be slower than pandas at 100k rows?
7. How is drift different from overfitting?
8. When should you avoid random train/test splits?

**Next skill after this notebook:** MLflow experiment tracking, proper data validation, Spark Structured Streaming and deployment/monitoring of inference pipelines.